# Fisher–KPP on Graphs — Traveling Wave Speed

This notebook simulates the **Fisher–KPP equation**

$$\partial_t u = \partial_{xx} u + u(1-u)$$

on three graph geometries (**Star**, **Extended Star**, **Tadpole**) and
computes the **traveling wave speed** $c$ on the outgoing edge.

**Method.** For each graph and each initial condition:
1. Time-step the PDE on every edge with explicit Euler + central differences,
2. After each step enforce the Neumann–Kirchhoff vertex condition
   $U_v = \frac{1}{d}\sum_k u^{(k)}_{\text{adj}}$,
3. Track the level-set position $x_{0.5}(t)$ on the outgoing edge,
4. Fit $x_{0.5}(t) = c\,t + b$ for $t \geq T_\text{START}$ — the slope $c$ is the wave speed.

**Expected speed** (KPP linear theory):
$c(\lambda) = \max(2,\, \lambda + 1/\lambda)$, with asymptotic $c^\star = 2$.


## Cell 1 — Imports & parameters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from mpl_toolkits.mplot3d import Axes3D
import os, warnings
warnings.filterwarnings('ignore')

# ── Spatial grid ─────────────────────────────────────────────────────────
L      = 100.0   # leg length
L_MID  = 12.0    # middle-edge length
DX     = 0.05    # grid spacing
N      = int(round(L     / DX))   # = 2000
N_MID  = int(round(L_MID / DX))   # = 240

# ── Time integration ─────────────────────────────────────────────────────
DT     = 0.001   # stability bound: DT < DX^2/2 = 0.00125
T_END  = 45.0
N_SNAP = 90      # snapshot count

# ── Speed-fit window ─────────────────────────────────────────────────────
# Don't start the speed calculation from t=0: the front needs ~3 time units
# to leave the vertex region. Use an absolute cutoff t = T_START.
T_START = 3.0

# ── Graph shape and initial-tail decay rate ──────────────────────────────
N_IN  = 2
N_OUT = 3
LAM   = 1.0      # used for Type 2 initial data: u0 = exp(-LAM * x)

# Coordinate arrays
#   incoming edge x in [-L, 0],  outgoing edge x in [0, L],  middle edge x in [0, L_MID]
X_IN  = np.linspace(-L,    0,      N     + 1)
X_OUT = np.linspace( 0,    L,      N     + 1)
X_MID = np.linspace( 0,    L_MID,  N_MID + 1)

os.makedirs('output', exist_ok=True)
print(f"Grid : DX={DX},  N={N},  N_MID={N_MID}")
print(f"Time : DT={DT},  T_END={T_END},  CFL={DT/DX**2:.3f}  (must be < 0.5)")
print(f"Speed fit window : t in [{T_START}, {T_END}]")
print(f"Graph: n={N_IN} incoming,  m={N_OUT} outgoing,  lam={LAM}")

## Cell 2 — Edge RHS (Fisher–KPP on one 1-D edge)

Each edge obeys $\partial_t u = \partial_{xx} u + u(1-u)$.

**Interior:** central differences. **Outer Neumann:** ghost-point trick.
**Vertex end:** RHS set to 0; the value is overridden algebraically by the
Kirchhoff coupling each step.

In [ ]:
def edge_rhs(u, dx, left='neumann', right='neumann'):
    f = np.empty_like(u)
    f[1:-1] = (u[:-2] - 2.0*u[1:-1] + u[2:]) / dx**2 + u[1:-1] * (1.0 - u[1:-1])
    f[0]  = 2.0*(u[1]  - u[0])  / dx**2 + u[0]  * (1.0 - u[0])  if left  == 'neumann' else 0.0
    f[-1] = 2.0*(u[-2] - u[-1]) / dx**2 + u[-1] * (1.0 - u[-1]) if right == 'neumann' else 0.0
    return f

print("edge_rhs defined  ok")

## Cell 3 — Graph 1: Star Graph simulator

**Kirchhoff at the single vertex:**
$(n+m)\,U_v = \sum_i u_i[N\!-\!1] + \sum_j u_j[1]$

In [ ]:
def simulate_star(n, m, U_in_0, U_out_0, T=T_END, dt=DT, verbose=True):
    U_in  = [u.copy() for u in U_in_0]
    U_out = [u.copy() for u in U_out_0]
    Uv0 = (sum(u[-2] for u in U_in) + sum(u[1] for u in U_out)) / (n + m)
    for u in U_in:  u[-1] = Uv0
    for u in U_out: u[ 0] = Uv0

    snap_times = np.linspace(0, T, N_SNAP + 1)
    snapshots  = []
    t = 0.0;  si = 0;  step = 0
    report = max(1, int(T / dt) // 4)

    while t <= T + 1e-10:
        if step == 0 or (si <= N_SNAP and t >= snap_times[si] - 1e-12):
            snapshots.append((t, [u.copy() for u in U_in], [u.copy() for u in U_out]))
            si = min(si + 1, N_SNAP)
        if t >= T:
            break
        dU_in  = [edge_rhs(u, DX, left='neumann', right='vertex') for u in U_in]
        dU_out = [edge_rhs(u, DX, left='vertex',  right='neumann') for u in U_out]
        U_in  = [U_in[k]  + dt * dU_in[k]  for k in range(n)]
        U_out = [U_out[k] + dt * dU_out[k] for k in range(m)]
        Uv = (sum(u[-2] for u in U_in) + sum(u[1] for u in U_out)) / (n + m)
        for u in U_in:  u[-1] = Uv
        for u in U_out: u[ 0] = Uv
        t += dt;  step += 1
        if verbose and step % report == 0:
            print(f"  t={t:.1f}/{T:.0f}  Uv={Uv:.4f}")
    return snapshots

print("simulate_star defined  ok")

## Cell 4 — Graph 2: Extended Star Graph simulator

**Left vertex:**  $(n+1)\,U_{vL} = \sum_i u_i[N\!-\!1] + u_\text{mid}[1]$

**Right vertex:** $(m+1)\,U_{vR} = u_\text{mid}[N_m\!-\!1] + \sum_j u_j[1]$

In [ ]:
def simulate_extended_star(n, m, U_in_0, U_mid_0, U_out_0, T=T_END, dt=DT, verbose=True):
    U_in  = [u.copy() for u in U_in_0]
    U_mid = U_mid_0.copy()
    U_out = [u.copy() for u in U_out_0]
    UvL0 = (sum(u[-2] for u in U_in) + U_mid[1])  / (n + 1)
    UvR0 = (U_mid[-2] + sum(u[1] for u in U_out)) / (m + 1)
    for u in U_in: u[-1]  = UvL0
    U_mid[0] = UvL0;  U_mid[-1] = UvR0
    for u in U_out: u[0]  = UvR0

    snap_times = np.linspace(0, T, N_SNAP + 1)
    snapshots  = []
    t = 0.0;  si = 0;  step = 0
    report = max(1, int(T / dt) // 4)

    while t <= T + 1e-10:
        if step == 0 or (si <= N_SNAP and t >= snap_times[si] - 1e-12):
            snapshots.append((t, [u.copy() for u in U_in], U_mid.copy(), [u.copy() for u in U_out]))
            si = min(si + 1, N_SNAP)
        if t >= T:
            break
        dU_in  = [edge_rhs(u,    DX, left='neumann', right='vertex') for u in U_in]
        dU_mid = edge_rhs(U_mid, DX, left='vertex',  right='vertex')
        dU_out = [edge_rhs(u,    DX, left='vertex',  right='neumann') for u in U_out]
        U_in  = [U_in[k]  + dt * dU_in[k]  for k in range(n)]
        U_mid = U_mid + dt * dU_mid
        U_out = [U_out[k] + dt * dU_out[k] for k in range(m)]
        UvL = (sum(u[-2] for u in U_in) + U_mid[1])  / (n + 1)
        for u in U_in: u[-1] = UvL
        U_mid[0] = UvL
        UvR = (U_mid[-2] + sum(u[1] for u in U_out)) / (m + 1)
        U_mid[-1] = UvR
        for u in U_out: u[0] = UvR
        t += dt;  step += 1
        if verbose and step % report == 0:
            print(f"  t={t:.1f}/{T:.0f}  vL={UvL:.4f}  vR={UvR:.4f}")
    return snapshots

print("simulate_extended_star defined  ok")

## Cell 5 — Graph 3: Tadpole Graph simulator

**Left vertex (1 in, 2 loop):**
$3\,U_{vL} = u_\text{in}[N\!-\!1] + u_{m1}[1] + u_{m2}[1]$

**Right vertex (2 loop, 1 out):**
$3\,U_{vR} = u_{m1}[N\!-\!1] + u_{m2}[N\!-\!1] + u_\text{out}[1]$

In [ ]:
def simulate_tadpole(u_in_0, u_m1_0, u_m2_0, u_out_0, T=T_END, dt=DT, verbose=True):
    u_in  = u_in_0.copy();  u_m1  = u_m1_0.copy()
    u_m2  = u_m2_0.copy();  u_out = u_out_0.copy()
    UvL0 = (u_in[-2]  + u_m1[1]  + u_m2[1])  / 3.0
    UvR0 = (u_m1[-2]  + u_m2[-2] + u_out[1]) / 3.0
    u_in[-1] = UvL0
    u_m1[0]  = UvL0;  u_m1[-1] = UvR0
    u_m2[0]  = UvL0;  u_m2[-1] = UvR0
    u_out[0] = UvR0

    snap_times = np.linspace(0, T, N_SNAP + 1)
    snapshots  = []
    t = 0.0;  si = 0;  step = 0
    report = max(1, int(T / dt) // 4)

    while t <= T + 1e-10:
        if step == 0 or (si <= N_SNAP and t >= snap_times[si] - 1e-12):
            snapshots.append((t, u_in.copy(), u_m1.copy(), u_m2.copy(), u_out.copy()))
            si = min(si + 1, N_SNAP)
        if t >= T:
            break
        du_in  = edge_rhs(u_in,  DX, left='neumann', right='vertex')
        du_m1  = edge_rhs(u_m1,  DX, left='vertex',  right='vertex')
        du_m2  = edge_rhs(u_m2,  DX, left='vertex',  right='vertex')
        du_out = edge_rhs(u_out, DX, left='vertex',  right='neumann')
        u_in  += dt * du_in;   u_m1 += dt * du_m1
        u_m2  += dt * du_m2;  u_out += dt * du_out
        UvL = (u_in[-2] + u_m1[1] + u_m2[1]) / 3.0
        u_in[-1] = UvL;  u_m1[0] = UvL;  u_m2[0] = UvL
        UvR = (u_m1[-2] + u_m2[-2] + u_out[1]) / 3.0
        u_m1[-1] = UvR;  u_m2[-1] = UvR;  u_out[0] = UvR
        t += dt;  step += 1
        if verbose and step % report == 0:
            print(f"  t={t:.1f}/{T:.0f}  vL={UvL:.4f}  vR={UvR:.4f}")
    return snapshots

print("simulate_tadpole defined  ok")

## Cell 6 — Initial data

| Type | Incoming | Outgoing / middle |
|------|----------|-------------------|
| **Type 1** (step) | $u_0 = 1$ | $u_0 = 0$ |
| **Type 2** (exp.) | $u_0 = 1$ | $u_0 = e^{-\lambda x}$ |

In [ ]:
def make_star_type1(n, m):
    return [np.ones(N+1) for _ in range(n)], [np.zeros(N+1) for _ in range(m)]

def make_star_type2(n, m, lam):
    return [np.ones(N+1) for _ in range(n)], [np.exp(-lam * X_OUT) for _ in range(m)]

def make_ext_type1(n, m):
    return ([np.ones(N+1)  for _ in range(n)],
            np.full(N_MID+1, 0.5),
            [np.zeros(N+1) for _ in range(m)])

def make_ext_type2(n, m, lam):
    return ([np.ones(N+1) for _ in range(n)],
            np.exp(-lam * X_MID),
            [np.exp(-lam * (L_MID + X_OUT)) for _ in range(m)])

def make_tad_type1():
    return np.ones(N+1), np.full(N_MID+1, 0.5), np.full(N_MID+1, 0.5), np.zeros(N+1)

def make_tad_type2(lam):
    return (np.ones(N+1), np.exp(-lam * X_MID),
            np.exp(-lam * X_MID), np.exp(-lam * (L_MID + X_OUT)))

print("Initial data factories defined  ok")

## Cell 7 — Speed estimator and 3D plotting utilities

`level_set_speed` tracks $x_{0.5}(t)$ on the outgoing edge using sub-grid
linear interpolation, then fits a line to $(t, x_{0.5})$ for $t\geq T_\text{START}$
and returns the slope.

`expected_speed` returns the KPP linear-theory prediction
$c(\lambda) = \max(2,\, \lambda + 1/\lambda)$.

The three `plot_*_3d` functions overlay several time steps on a **single**
3D axes. Color advances with time (viridis colormap, see colorbar) and a
black dot marks each vertex at the final time. Watching the family of curves
sweep along the outgoing leg gives a direct visual sense of the wave speed.

In [ ]:
def level_set_speed(snapshots, get_out, threshold=0.5, t_start=None):
    """Linear-fit wave-speed estimator.

    1. At each snapshot, find sub-grid x where u crosses `threshold`.
    2. Fit a straight line to (t, x_front) using only t >= t_start.
    3. Return (times, positions, slope).  The slope IS the wave speed c.
    """
    if t_start is None:
        t_start = T_START
    times = [];  positions = []
    for snap in snapshots:
        t = snap[0]
        u = get_out(snap)
        if u.max() <= threshold or u.min() >= threshold:
            continue
        idx = np.where(u <= threshold)[0]
        if len(idx) == 0 or idx[0] == 0:
            continue
        i = idx[0]
        x_front = X_OUT[i-1] + (threshold - u[i-1]) / (u[i] - u[i-1]) * DX
        times.append(t);  positions.append(x_front)
    if len(times) < 4:
        return np.array(times), np.array(positions), np.nan
    times     = np.array(times)
    positions = np.array(positions)
    mask      = times >= t_start
    if mask.sum() < 3:
        mask = np.ones(len(times), dtype=bool)
    speed = np.polyfit(times[mask], positions[mask], 1)[0]
    return times, positions, speed


def expected_speed(lam):
    """KPP expected speed:  c(lam) = max(2, lam + 1/lam)."""
    return np.maximum(2.0, lam + 1.0 / lam)


# ── 3D plotting utilities (overlay style) ────────────────────────────────
TIME_CMAP = plt.cm.viridis

def _find_snaps(snapshots, times_to_plot):
    all_t = np.array([s[0] for s in snapshots])
    return [int(np.argmin(np.abs(all_t - t))) for t in times_to_plot]

def _style_ax3d(ax, title):
    ax.set_title(title, fontsize=13, fontweight='bold', pad=10)
    ax.set_xlabel('x', labelpad=4)
    ax.set_ylabel('y', labelpad=4)
    ax.set_zlabel('u', labelpad=4)
    ax.set_zlim(-0.05, 1.15)
    ax.tick_params(labelsize=8)
    ax.view_init(elev=25, azim=-60)

def _vertex_dot(ax, vx, vy, u_val):
    ax.scatter([vx], [vy], [u_val], color='black', s=70, zorder=10, depthshade=False)

def _add_time_colorbar(fig, ax, t_min, t_max):
    """Colorbar mapping the viridis colormap to time."""
    sm = plt.cm.ScalarMappable(cmap=TIME_CMAP,
                                norm=plt.Normalize(vmin=t_min, vmax=t_max))
    sm.set_array([])
    cb = fig.colorbar(sm, ax=ax, shrink=0.55, pad=0.10, label='time  t')
    cb.ax.tick_params(labelsize=8)


# ── Star geometry & overlay 3D plot ──────────────────────────────────────
def _star_geometry(n, m, L_edge=L):
    v = np.zeros(2)
    in_angs  = np.linspace(120, 240, n) if n > 1 else np.array([180.0])
    out_angs = np.linspace(-60, 60, m)  if m > 1 else np.array([0.0])
    edges = []
    for ang in in_angs:
        r = np.radians(ang);  d = np.array([np.cos(r), np.sin(r)])
        xs = v[0] + L_edge * d[0] * np.linspace(1, 0, N+1)
        ys = v[1] + L_edge * d[1] * np.linspace(1, 0, N+1)
        edges.append(('in', xs, ys))
    for ang in out_angs:
        r = np.radians(ang);  d = np.array([np.cos(r), np.sin(r)])
        xs = v[0] + L_edge * d[0] * np.linspace(0, 1, N+1)
        ys = v[1] + L_edge * d[1] * np.linspace(0, 1, N+1)
        edges.append(('out', xs, ys))
    return v, edges

def plot_star_3d(snapshots, n, m, times_to_plot,
                 title='Star Graph', fname=None):
    v, edges = _star_geometry(n, m)
    idxs   = _find_snaps(snapshots, times_to_plot)
    actual = [snapshots[i][0] for i in idxs]

    fig = plt.figure(figsize=(11, 8.5))
    ax  = fig.add_subplot(111, projection='3d')
    _style_ax3d(ax, title)

    colors = TIME_CMAP(np.linspace(0.1, 0.9, len(idxs)))
    for ti, (si, color, t_val) in enumerate(zip(idxs, colors, actual)):
        snap = snapshots[si]
        U_in, U_out = snap[1], snap[2]
        alpha = 0.45 + 0.55 * (ti / max(len(idxs)-1, 1))
        lw    = 1.6  + 1.0  * (ti / max(len(idxs)-1, 1))
        in_i = out_i = 0
        for etype, xs, ys in edges:
            if etype == 'in':
                u_vals = U_in[in_i];  in_i += 1
            else:
                u_vals = U_out[out_i];  out_i += 1
            ax.plot(xs, ys, u_vals, color=color, lw=lw, alpha=alpha)
        # Vertex dot at final time only
        if ti == len(idxs) - 1:
            _vertex_dot(ax, v[0], v[1], U_in[0][-1])

    _add_time_colorbar(fig, ax, actual[0], actual[-1])
    plt.tight_layout()
    if fname: fig.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()


# ── Extended Star geometry & overlay 3D plot ─────────────────────────────
def _ext_geometry(n, m, L_edge=L, Lm=L_MID):
    vL = np.array([0.0, 0.0]);  vR = np.array([Lm, 0.0])
    in_angs  = np.linspace(120, 240, n) if n > 1 else np.array([180.0])
    out_angs = np.linspace(-60, 60, m)  if m > 1 else np.array([0.0])
    edges = []
    for ang in in_angs:
        r = np.radians(ang);  d = np.array([np.cos(r), np.sin(r)])
        xs = vL[0] + L_edge * d[0] * np.linspace(1, 0, N+1)
        ys = vL[1] + L_edge * d[1] * np.linspace(1, 0, N+1)
        edges.append(('in', xs, ys))
    edges.append(('mid',
                  np.linspace(vL[0], vR[0], N_MID+1),
                  np.zeros(N_MID+1)))
    for ang in out_angs:
        r = np.radians(ang);  d = np.array([np.cos(r), np.sin(r)])
        xs = vR[0] + L_edge * d[0] * np.linspace(0, 1, N+1)
        ys = vR[1] + L_edge * d[1] * np.linspace(0, 1, N+1)
        edges.append(('out', xs, ys))
    return vL, vR, edges

def plot_extended_star_3d(snapshots, n, m, times_to_plot,
                          title='Extended Star', fname=None):
    vL, vR, edges = _ext_geometry(n, m)
    idxs   = _find_snaps(snapshots, times_to_plot)
    actual = [snapshots[i][0] for i in idxs]

    fig = plt.figure(figsize=(11, 8.5))
    ax  = fig.add_subplot(111, projection='3d')
    _style_ax3d(ax, title)

    colors = TIME_CMAP(np.linspace(0.1, 0.9, len(idxs)))
    for ti, (si, color, t_val) in enumerate(zip(idxs, colors, actual)):
        snap = snapshots[si]
        U_in, U_mid, U_out = snap[1], snap[2], snap[3]
        alpha = 0.45 + 0.55 * (ti / max(len(idxs)-1, 1))
        lw    = 1.6  + 1.0  * (ti / max(len(idxs)-1, 1))
        in_i = out_i = 0
        for etype, xs, ys in edges:
            if etype == 'in':
                u_vals = U_in[in_i];  in_i += 1
            elif etype == 'mid':
                u_vals = U_mid
            else:
                u_vals = U_out[out_i];  out_i += 1
            ax.plot(xs, ys, u_vals, color=color, lw=lw, alpha=alpha)
        if ti == len(idxs) - 1:
            _vertex_dot(ax, vL[0], vL[1], U_in[0][-1])
            _vertex_dot(ax, vR[0], vR[1], U_out[0][0])

    _add_time_colorbar(fig, ax, actual[0], actual[-1])
    plt.tight_layout()
    if fname: fig.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()


# ── Tadpole geometry & overlay 3D plot ───────────────────────────────────
def _tadpole_geometry(L_edge=L, Lm=L_MID):
    r = Lm / 2.0
    vL = np.array([-r, 0.0]);  vR = np.array([ r, 0.0])
    xs_in = np.linspace(vL[0] - L_edge, vL[0], N+1);  ys_in = np.zeros(N+1)
    theta1 = np.linspace(np.pi, 0, N_MID+1)
    xs_m1  = r * np.cos(theta1);  ys_m1 = r * np.sin(theta1)
    theta2 = np.linspace(np.pi, 2*np.pi, N_MID+1)
    xs_m2  = r * np.cos(theta2);  ys_m2 = r * np.sin(theta2)
    xs_out = np.linspace(vR[0], vR[0] + L_edge, N+1);  ys_out = np.zeros(N+1)
    return vL, vR, xs_in, ys_in, xs_m1, ys_m1, xs_m2, ys_m2, xs_out, ys_out

def plot_tadpole_3d(snapshots, times_to_plot,
                    title='Tadpole Graph', fname=None):
    geom = _tadpole_geometry()
    vL, vR = geom[0], geom[1]
    xs_in, ys_in   = geom[2], geom[3]
    xs_m1, ys_m1   = geom[4], geom[5]
    xs_m2, ys_m2   = geom[6], geom[7]
    xs_out, ys_out = geom[8], geom[9]

    idxs   = _find_snaps(snapshots, times_to_plot)
    actual = [snapshots[i][0] for i in idxs]

    fig = plt.figure(figsize=(11, 8.5))
    ax  = fig.add_subplot(111, projection='3d')
    _style_ax3d(ax, title)

    colors = TIME_CMAP(np.linspace(0.1, 0.9, len(idxs)))
    for ti, (si, color, t_val) in enumerate(zip(idxs, colors, actual)):
        snap = snapshots[si]
        u_in, u_m1, u_m2, u_out = snap[1], snap[2], snap[3], snap[4]
        alpha = 0.45 + 0.55 * (ti / max(len(idxs)-1, 1))
        lw    = 1.6  + 1.0  * (ti / max(len(idxs)-1, 1))
        ax.plot(xs_in,  ys_in,  u_in,  color=color, lw=lw, alpha=alpha)
        ax.plot(xs_m1,  ys_m1,  u_m1,  color=color, lw=lw, alpha=alpha)
        ax.plot(xs_m2,  ys_m2,  u_m2,  color=color, lw=lw, alpha=alpha)
        ax.plot(xs_out, ys_out, u_out, color=color, lw=lw, alpha=alpha)
        if ti == len(idxs) - 1:
            _vertex_dot(ax, vL[0], vL[1], u_in[-1])
            _vertex_dot(ax, vR[0], vR[1], u_out[0])

    _add_time_colorbar(fig, ax, actual[0], actual[-1])
    plt.tight_layout()
    if fname: fig.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()


print("level_set_speed, expected_speed, plot_*_3d (overlay)  defined  ok")

## Cell 8 — Star Graph: simulate, plot, measure speed

In [ ]:
print("=" * 55)
print(f"  Star Graph  (n={N_IN} incoming, m={N_OUT} outgoing)")
print("=" * 55)

# Simulate Type 1 and Type 2
print("\n--- Type 1: step initial data ---")
snap_star1 = simulate_star(N_IN, N_OUT, *make_star_type1(N_IN, N_OUT))
plot_star_3d(snap_star1, N_IN, N_OUT,
             times_to_plot=np.linspace(T_START, T_END, 5),
             title=f'Star Graph (n={N_IN}, m={N_OUT})  |  Type 1: step initial',
             fname='output/star_type1_3d.png')

print(f"\n--- Type 2: u0 = exp(-{LAM}*x) ---")
snap_star2 = simulate_star(N_IN, N_OUT, *make_star_type2(N_IN, N_OUT, LAM))
plot_star_3d(snap_star2, N_IN, N_OUT,
             times_to_plot=np.linspace(T_START, T_END, 5),
             title=f'Star Graph (n={N_IN}, m={N_OUT})  |  Type 2: exp(-{LAM}*x)',
             fname='output/star_type2_3d.png')

# Outgoing-edge accessor for level_set_speed
get_star = lambda snap: snap[2][0]

# Speed
_, _, c_star1 = level_set_speed(snap_star1, get_star)
_, _, c_star2 = level_set_speed(snap_star2, get_star)
print(f"\n  Type 1 (step)        : c = {c_star1:.4f}   (expected c* = 2.0000)")
print(f"  Type 2 (lam={LAM})       : c = {c_star2:.4f}   (expected = {expected_speed(LAM):.4f})")

## Cell 9 — Extended Star Graph: simulate, plot, measure speed

In [ ]:
print("=" * 55)
print(f"  Extended Star  (n={N_IN}, middle, m={N_OUT})")
print("=" * 55)

print("\n--- Type 1: step initial data ---")
snap_ext1 = simulate_extended_star(N_IN, N_OUT, *make_ext_type1(N_IN, N_OUT))
plot_extended_star_3d(snap_ext1, N_IN, N_OUT,
             times_to_plot=np.linspace(T_START, T_END, 5),
             title=f'Extended Star (n={N_IN}, m={N_OUT})  |  Type 1: step initial',
             fname='output/extended_type1_3d.png')

print(f"\n--- Type 2: u0 = exp(-{LAM}*x) ---")
snap_ext2 = simulate_extended_star(N_IN, N_OUT, *make_ext_type2(N_IN, N_OUT, LAM))
plot_extended_star_3d(snap_ext2, N_IN, N_OUT,
             times_to_plot=np.linspace(T_START, T_END, 5),
             title=f'Extended Star (n={N_IN}, m={N_OUT})  |  Type 2: exp(-{LAM}*x)',
             fname='output/extended_type2_3d.png')

get_ext = lambda snap: snap[3][0]

_, _, c_ext1 = level_set_speed(snap_ext1, get_ext)
_, _, c_ext2 = level_set_speed(snap_ext2, get_ext)
print(f"\n  Type 1 (step)        : c = {c_ext1:.4f}   (expected c* = 2.0000)")
print(f"  Type 2 (lam={LAM})       : c = {c_ext2:.4f}   (expected = {expected_speed(LAM):.4f})")

## Cell 10 — Tadpole Graph: simulate, plot, measure speed

In [ ]:
print("=" * 55)
print("  Tadpole Graph  (1 in, 2 loop, 1 out)")
print("=" * 55)

print("\n--- Type 1: step initial data ---")
snap_tad1 = simulate_tadpole(*make_tad_type1())
plot_tadpole_3d(snap_tad1,
             times_to_plot=np.linspace(T_START, T_END, 5),
             title='Tadpole Graph  |  Type 1: step initial',
             fname='output/tadpole_type1_3d.png')

print(f"\n--- Type 2: u0 = exp(-{LAM}*x) ---")
snap_tad2 = simulate_tadpole(*make_tad_type2(LAM))
plot_tadpole_3d(snap_tad2,
             times_to_plot=np.linspace(T_START, T_END, 5),
             title=f'Tadpole Graph  |  Type 2: exp(-{LAM}*x)',
             fname='output/tadpole_type2_3d.png')

get_tad = lambda snap: snap[4]

_, _, c_tad1 = level_set_speed(snap_tad1, get_tad)
_, _, c_tad2 = level_set_speed(snap_tad2, get_tad)
print(f"\n  Type 1 (step)        : c = {c_tad1:.4f}   (expected c* = 2.0000)")
print(f"  Type 2 (lam={LAM})       : c = {c_tad2:.4f}   (expected = {expected_speed(LAM):.4f})")

## Cell 11 — Final results summary

In [ ]:
print("=" * 72)
print("  FINAL RESULTS  —  Traveling Wave Speed")
print("=" * 72)
print(f"  {'Graph':<20} {'Initial':<22} {'c_simulated':>12} {'c_expected':>12} {'err':>7}")
print("  " + "-" * 70)
rows = [
    ('Star',          'Type 1 (step)',             c_star1, 2.0),
    ('Star',          f'Type 2 (lam={LAM})',       c_star2, expected_speed(LAM)),
    ('Extended Star', 'Type 1 (step)',             c_ext1,  2.0),
    ('Extended Star', f'Type 2 (lam={LAM})',       c_ext2,  expected_speed(LAM)),
    ('Tadpole',       'Type 1 (step)',             c_tad1,  2.0),
    ('Tadpole',       f'Type 2 (lam={LAM})',       c_tad2,  expected_speed(LAM)),
]
for graph, init, c_sim, c_exp in rows:
    err = abs(c_sim - c_exp)
    print(f"  {graph:<20} {init:<22} {c_sim:>12.4f} {c_exp:>12.4f} {err:>7.3f}")
print("=" * 72)
print()
print(f"  Expected speed formula:  c(lam) = max(2, lam + 1/lam),  c* = 2")
print(f"  Simulation:  L = {L},  T_END = {T_END},  T_START = {T_START},  DX = {DX}")